# Initial Dataset preprocessing

In [131]:
import os
import mne
import csv
import pandas as pd
import numpy as np
import json

In [132]:
parsed_path = "./../EEG_data/parsed/"
summary = "./../EEG_data/parsed/summary.csv"
with open(summary, 'r') as csvfile:
    csvreader = csv.DictReader(csvfile,delimiter =';')
    data_list = []
    for row in csvreader:
        data_list.append(row)
folders = []
for data in data_list:
    if (not data["session_id"] in folders):
        folders.append(data["session_id"])

# paths to the parsed dataset files
p_baseline_intervals = []
p_baseline_markers = []
p_baseline_metadata = []
p_baseline_samples = []
p_main_intervals = []
p_main_markers = []
p_main_metadata = []
p_main_samples = []

for folder in folders:
    p_baseline_intervals.append(parsed_path + folder + "/baseline_intervals.parquet")
    p_baseline_markers.append(parsed_path + folder + "/baseline_markers.parquet")
    p_baseline_metadata.append(parsed_path + folder + "/baseline_metadata.json")
    p_baseline_samples.append(parsed_path + folder + "/baseline_samples.parquet")

    p_main_intervals.append(parsed_path + folder + "/main_intervals.parquet")
    p_main_markers.append(parsed_path + folder + "/main_markers.parquet")
    p_main_metadata.append(parsed_path + folder + "/main_metadata.json")
    p_main_samples.append(parsed_path + folder + "/main_samples.parquet")


In [133]:
# MNE setup
# create raw objects

subject_num = len(folders)
baseline_raw = []
main_raw = []
for i in range (0, subject_num):
    baseline_intervals = pd.read_parquet(p_baseline_intervals[i])
    baseline_markers = pd.read_parquet(p_baseline_markers[i])
    baseline_samples = pd.read_parquet(p_baseline_samples[i])
   
    with open(p_baseline_metadata[i]) as b_meta:
        b_metadata = json.load(b_meta)
    bs_metadata = pd.json_normalize(b_metadata)
    

    main_intervals = pd.read_parquet(p_main_intervals[i])
    main_markers = pd.read_parquet(p_main_markers[i])
    main_samples = pd.read_parquet(p_main_samples[i])

    with open(p_main_metadata[i]) as m_meta:
        m_metadata = json.load(m_meta)
    mn_metadata = pd.json_normalize(m_metadata)

    print(folders[i] + " baseline")
    # baseline raw object
    # data: array, shape (n_channels, n_times)
    bs_channels = bs_metadata.header[0][:-2]
    bs_data = []
    for channel in bs_channels:
        bs_times = baseline_samples[channel].values
        bs_data.append(bs_times)
    # info
    bs_sfreq = bs_metadata.fs_est.values[0]
    print(bs_sfreq)
    bs_info = mne.create_info(bs_channels, bs_sfreq, 'eeg')
    # create a Raw object
    bs_raw = mne.io.RawArray(bs_data, bs_info, first_samp=0)
    baseline_raw.append(bs_raw)
    print(len(bs_raw))

    print(folders[i] + " main")
    # main raw object
    # data: array, shape (n_channels, n_times)
    mn_channels = mn_metadata.header[0][:-2]
    mn_data = []
    for channel in mn_channels:
        mn_times = main_samples[channel].values
        mn_data.append(mn_times)
    # info
    mn_sfreq = mn_metadata.fs_est.values[0]
    print(mn_sfreq)
    mn_info = mne.create_info(mn_channels, mn_sfreq, 'eeg')
    # create a Raw object
    mn_raw = mne.io.RawArray(mn_data, mn_info, first_samp=0)
    main_raw.append(mn_raw)
    print(len(mn_raw))



2026-04-04_Lazutkin baseline
118.0552995391705
Creating RawArray with float64 data, n_channels=18, n_times=150233
    Range : 0 ... 150232 =      0.000 ...  1272.556 secs
Ready.
150233
2026-04-04_Lazutkin main
248.43023255813952
Creating RawArray with float64 data, n_channels=18, n_times=22876
    Range : 0 ... 22875 =      0.000 ...    92.078 secs
Ready.
22876
2026-04-04_Vundirov baseline
249.6441605839416
Creating RawArray with float64 data, n_channels=18, n_times=150211
    Range : 0 ... 150210 =      0.000 ...   601.696 secs
Ready.
150211
2026-04-04_Vundirov main
251.9848484848485
Creating RawArray with float64 data, n_channels=18, n_times=17500
    Range : 0 ... 17499 =      0.000 ...    69.445 secs
Ready.
17500
2026-04-10_Aminov baseline
249.88704318936877
Creating RawArray with float64 data, n_channels=18, n_times=150168
    Range : 0 ... 150167 =      0.000 ...   600.940 secs
Ready.
150168
2026-04-10_Aminov main
250.1304347826087
Creating RawArray with float64 data, n_channels=

In [ ]:
# re-referencing
